# 📊 Paper Analysis Visualizations
## Epilepsy Literature Review - ECG/Wearable Seizure Detection & Prediction

This notebook provides visualizations and analysis of the 105 papers collected for the literature review.

In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Load data
df = pd.read_csv('paper_analysis_results.csv')
print(f"Loaded {len(df)} papers")
df.head()

## 1. Publication Timeline

In [ ]:
# Publications by year
year_counts = df['year'].dropna().astype(int).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(year_counts)))
bars = ax.bar(year_counts.index.astype(str), year_counts.values, color=colors, edgecolor='navy', linewidth=0.5)

# Add value labels on bars
for bar, val in zip(bars, year_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, str(val), 
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Number of Papers', fontsize=12)
ax.set_title('📅 Publications by Year', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('fig_publications_by_year.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📈 Trend: Papers from 2020-2025: {year_counts[year_counts.index >= 2020].sum()} ({year_counts[year_counts.index >= 2020].sum()/len(df)*100:.1f}%)")

## 2. Modalities Distribution

In [ ]:
# Count modalities
modality_counts = Counter()
for mods in df['modalities'].dropna():
    for mod in mods.split('; '):
        if mod.strip():
            modality_counts[mod.strip()] += 1

mods_df = pd.DataFrame(list(modality_counts.items()), columns=['Modality', 'Count'])
mods_df = mods_df.sort_values('Count', ascending=True)

# Horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if m == 'EEG' else '#3498db' if m in ['ECG', 'HRV'] else '#2ecc71' for m in mods_df['Modality']]
bars = ax.barh(mods_df['Modality'], mods_df['Count'], color=colors, edgecolor='white', linewidth=1)

# Add percentage labels
for bar, val in zip(bars, mods_df['Count']):
    pct = val / len(df) * 100
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, 
            f'{val} ({pct:.0f}%)', va='center', fontsize=10)

ax.set_xlabel('Number of Papers', fontsize=12)
ax.set_title('🔬 Modalities Mentioned in Papers', fontsize=14, fontweight='bold')
ax.set_xlim(0, max(mods_df['Count']) * 1.2)

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#e74c3c', label='EEG (often for comparison)'),
                   Patch(facecolor='#3498db', label='ECG/HRV (cardiac)'),
                   Patch(facecolor='#2ecc71', label='Other wearable')]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('fig_modalities.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Tasks: Detection vs Prediction

In [ ]:
# Count tasks
task_counts = Counter()
for tasks in df['tasks'].dropna():
    for task in tasks.split('; '):
        if task.strip():
            task_counts[task.strip()] += 1

# Pie chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors_pie = ['#3498db', '#e74c3c', '#f39c12']
tasks_labels = list(task_counts.keys())
tasks_values = list(task_counts.values())

wedges, texts, autotexts = axes[0].pie(tasks_values, labels=tasks_labels, autopct='%1.1f%%', 
                                        colors=colors_pie, startangle=90, explode=[0.02]*len(tasks_labels))
axes[0].set_title('🎯 Research Focus Distribution', fontsize=14, fontweight='bold')

# Bar comparison
task_df = pd.DataFrame({'Task': tasks_labels, 'Count': tasks_values})
task_df = task_df.sort_values('Count', ascending=False)
bars = axes[1].bar(task_df['Task'], task_df['Count'], color=colors_pie[:len(task_df)], edgecolor='white')
for bar, val in zip(bars, task_df['Count']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, str(val), 
                 ha='center', fontweight='bold')
axes[1].set_ylabel('Number of Papers')
axes[1].set_title('Papers by Task Type', fontsize=12)

plt.tight_layout()
plt.savefig('fig_tasks.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Machine Learning Methods

In [ ]:
# ML method breakdown
ml_data = {
    'Deep Learning': df['uses_deep_learning'].sum(),
    'Traditional ML': df['uses_traditional_ml'].sum(),
    'Both': ((df['uses_deep_learning'] == True) & (df['uses_traditional_ml'] == True)).sum(),
    'Neither': ((df['uses_deep_learning'] == False) & (df['uses_traditional_ml'] == False)).sum()
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
methods = ['Deep Learning', 'Traditional ML']
values = [ml_data['Deep Learning'], ml_data['Traditional ML']]
colors_ml = ['#9b59b6', '#1abc9c']
bars = axes[0].bar(methods, values, color=colors_ml, edgecolor='white', width=0.6)
for bar, val in zip(bars, values):
    pct = val / len(df) * 100
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                 f'{val}\n({pct:.0f}%)', ha='center', fontweight='bold')
axes[0].set_ylabel('Number of Papers')
axes[0].set_title('🤖 ML Methods Used', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, max(values) * 1.2)

# Venn-style breakdown
venn_labels = ['DL Only', 'TML Only', 'Both', 'Neither']
venn_values = [
    ml_data['Deep Learning'] - ml_data['Both'],
    ml_data['Traditional ML'] - ml_data['Both'],
    ml_data['Both'],
    ml_data['Neither']
]
colors_venn = ['#9b59b6', '#1abc9c', '#3498db', '#bdc3c7']
axes[1].pie(venn_values, labels=venn_labels, autopct='%1.0f%%', colors=colors_venn, startangle=90)
axes[1].set_title('ML Method Combinations', fontsize=12)

plt.tight_layout()
plt.savefig('fig_ml_methods.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. ECG/Wearable Focus Analysis

In [ ]:
# ECG-based and wearable papers
ecg_wearable_data = {
    'ECG/HRV-based': df['is_ecg_based'].sum(),
    'Wearable-focused': df['is_wearable'].sum(),
    'Both ECG & Wearable': ((df['is_ecg_based'] == True) & (df['is_wearable'] == True)).sum()
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
categories = list(ecg_wearable_data.keys())
values = list(ecg_wearable_data.values())
colors_cat = ['#e74c3c', '#2ecc71', '#3498db']

bars = axes[0].bar(categories, values, color=colors_cat, edgecolor='white', width=0.6)
for bar, val in zip(bars, values):
    pct = val / len(df) * 100
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                 f'{val} ({pct:.0f}%)', ha='center', fontweight='bold')
axes[0].set_ylabel('Number of Papers')
axes[0].set_title('📌 Papers Relevant to Your Research Focus', fontsize=14, fontweight='bold')
axes[0].set_ylim(0, max(values) * 1.2)
axes[0].tick_params(axis='x', rotation=15)

# Timeline of ECG papers
ecg_papers = df[df['is_ecg_based'] == True]
ecg_by_year = ecg_papers['year'].dropna().astype(int).value_counts().sort_index()

axes[1].plot(ecg_by_year.index, ecg_by_year.values, 'o-', color='#e74c3c', linewidth=2, markersize=8)
axes[1].fill_between(ecg_by_year.index, ecg_by_year.values, alpha=0.3, color='#e74c3c')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Number of ECG/HRV Papers')
axes[1].set_title('📈 ECG/HRV-based Papers Over Time', fontsize=12)

plt.tight_layout()
plt.savefig('fig_ecg_wearable.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n🎯 Core papers for your review (ECG/HRV + Wearable): {ecg_wearable_data['Both ECG & Wearable']}")

## 6. Datasets Used

In [ ]:
# Count datasets
dataset_counts = Counter()
for datasets in df['datasets'].dropna():
    for ds in datasets.split('; '):
        if ds.strip():
            dataset_counts[ds.strip()] += 1

# Filter to top datasets
top_datasets = dict(dataset_counts.most_common(12))

fig, ax = plt.subplots(figsize=(12, 6))

# Categorize datasets
public_datasets = ['CHB-MIT', 'CHBMIT', 'Physionet', 'Siena', 'Epilepsiae', 'European Epilepsy Database', 'Freiburg', 'TUH', 'Temple', 'TUSZ']
colors_ds = ['#27ae60' if ds in public_datasets else '#e67e22' if ds == 'wearable' else '#3498db' for ds in top_datasets.keys()]

bars = ax.barh(list(top_datasets.keys()), list(top_datasets.values()), color=colors_ds, edgecolor='white')
for bar, val in zip(bars, top_datasets.values()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, str(val), va='center')

ax.set_xlabel('Number of Papers')
ax.set_title('💾 Datasets Mentioned in Papers', fontsize=14, fontweight='bold')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#27ae60', label='Public EEG datasets'),
                   Patch(facecolor='#e67e22', label='Wearable data'),
                   Patch(facecolor='#3498db', label='Clinical/EMU data')]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('fig_datasets.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Evaluation Methods

In [ ]:
# Count evaluation methods
eval_counts = Counter()
for evals in df['evaluation'].dropna():
    for ev in evals.split('; '):
        if ev.strip():
            eval_counts[ev.strip()] += 1

fig, ax = plt.subplots(figsize=(10, 5))

eval_df = pd.DataFrame(list(eval_counts.items()), columns=['Method', 'Count'])
eval_df = eval_df.sort_values('Count', ascending=False)

colors_eval = ['#2ecc71', '#3498db', '#e74c3c']
bars = ax.bar(eval_df['Method'], eval_df['Count'], color=colors_eval[:len(eval_df)], edgecolor='white')
for bar, val in zip(bars, eval_df['Count']):
    pct = val / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{val} ({pct:.0f}%)', ha='center', fontweight='bold')

ax.set_ylabel('Number of Papers')
ax.set_title('✅ Evaluation Methods Used', fontsize=14, fontweight='bold')
ax.set_xticklabels(['Patient-Specific', 'Cross-Validation', 'Patient-Independent'], rotation=0)

plt.tight_layout()
plt.savefig('fig_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n⚠️ Note: Only 4 papers use patient-independent evaluation - a key limitation in the field!")

## 8. Summary Dashboard

In [ ]:
# Create a summary dashboard
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('📊 Literature Review Summary Dashboard', fontsize=16, fontweight='bold', y=1.02)

# 1. Total papers and key counts
metrics = {
    'Total Papers': len(df),
    'ECG/HRV Papers': df['is_ecg_based'].sum(),
    'Wearable Papers': df['is_wearable'].sum(),
    'Detection': task_counts.get('detection', 0),
    'Prediction': task_counts.get('prediction', 0)
}
colors_m = ['#34495e', '#e74c3c', '#2ecc71', '#3498db', '#9b59b6']
bars = axes[0, 0].bar(metrics.keys(), metrics.values(), color=colors_m)
for bar, val in zip(bars, metrics.values()):
    axes[0, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(val), ha='center', fontweight='bold')
axes[0, 0].set_title('Key Counts', fontweight='bold')
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Years (recent 10)
recent_years = year_counts[year_counts.index >= 2015]
axes[0, 1].bar(recent_years.index.astype(str), recent_years.values, color='#3498db')
axes[0, 1].set_title('Publications 2015-2025', fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Modalities pie
mod_for_pie = dict(list(modality_counts.most_common(5)))
axes[0, 2].pie(mod_for_pie.values(), labels=mod_for_pie.keys(), autopct='%1.0f%%', startangle=90)
axes[0, 2].set_title('Modalities', fontweight='bold')

# 4. ML Methods
ml_for_bar = {'Deep Learning': ml_data['Deep Learning'], 'Traditional ML': ml_data['Traditional ML']}
axes[1, 0].bar(ml_for_bar.keys(), ml_for_bar.values(), color=['#9b59b6', '#1abc9c'])
axes[1, 0].set_title('ML Methods', fontweight='bold')

# 5. Study types
study_counts = Counter()
for studies in df['study_types'].dropna():
    for st in studies.split('; '):
        if st.strip():
            study_counts[st.strip()] += 1
top_studies = dict(study_counts.most_common(5))
axes[1, 1].barh(list(top_studies.keys()), list(top_studies.values()), color='#e67e22')
axes[1, 1].set_title('Study Types', fontweight='bold')

# 6. Key insights text
axes[1, 2].axis('off')
insights = f"""📌 KEY INSIGHTS

• {len(df)} papers analyzed
• {df['is_ecg_based'].sum()} papers ({df['is_ecg_based'].sum()/len(df)*100:.0f}%) use ECG/HRV
• {df['is_wearable'].sum()} papers ({df['is_wearable'].sum()/len(df)*100:.0f}%) focus on wearables
• Detection ({task_counts.get('detection', 0)}) > Prediction ({task_counts.get('prediction', 0)})
• Deep Learning dominant ({ml_data['Deep Learning']}/{len(df)} papers)
• Only {eval_counts.get('patient_independent', 0)} papers use patient-independent eval
• Peak publication year: 2021 ({year_counts.get(2021, 0)} papers)
"""
axes[1, 2].text(0.1, 0.5, insights, fontsize=11, verticalalignment='center', 
                bbox=dict(boxstyle='round', facecolor='#ecf0f1', alpha=0.8))

plt.tight_layout()
plt.savefig('fig_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Filter Papers for Your Review Scope

In [ ]:
# Filter to papers most relevant for ECG-based seizure prediction/detection review
# Criteria: ECG/HRV-based AND (detection OR prediction task)

relevant_df = df[
    (df['is_ecg_based'] == True) & 
    (df['tasks'].str.contains('detection|prediction', case=False, na=False))
].copy()

print(f"📚 Papers matching your review scope (ECG/HRV + detection/prediction): {len(relevant_df)}")
print(f"   This is {len(relevant_df)/len(df)*100:.1f}% of all papers\n")

# Show summary
print("Top relevant papers by word count (likely most detailed):")
top_relevant = relevant_df.nlargest(10, 'word_count')[['title', 'authors', 'year', 'primary_task', 'word_count']]
display(top_relevant)

In [ ]:
# Export filtered list
relevant_df.to_csv('papers_ecg_seizure_relevant.csv', index=False)
print(f"✅ Exported {len(relevant_df)} relevant papers to 'papers_ecg_seizure_relevant.csv'")

# Show breakdown
print(f"\n📊 Breakdown of relevant papers:")
print(f"   Detection-focused: {relevant_df['primary_task'].str.contains('detection', case=False, na=False).sum()}")
print(f"   Prediction-focused: {relevant_df['primary_task'].str.contains('prediction', case=False, na=False).sum()}")
print(f"   Wearable-based: {relevant_df['is_wearable'].sum()}")
print(f"   Uses Deep Learning: {relevant_df['uses_deep_learning'].sum()}")

## 10. Papers Needing Manual Review

In [ ]:
# Papers that might be outside scope or need manual review
no_task = df[df['tasks'].isna() | (df['tasks'] == '')]
no_modality = df[df['modalities'].isna() | (df['modalities'] == '')]

print(f"⚠️ Papers without clear task (detection/prediction): {len(no_task)}")
print(f"⚠️ Papers without clear modality: {len(no_modality)}")

# These might be background papers, methodology papers, or incorrectly parsed
review_needed = df[
    (df['tasks'].isna() | (df['tasks'] == '')) |
    (df['modalities'].isna() | (df['modalities'] == ''))
][['filename', 'title', 'year']]

print(f"\n📋 Papers needing manual review ({len(review_needed)}):")
display(review_needed)